# ⚓ BITACORA DE RESPUESTAS — Capitulo IIICodigo pandas de todos los cruces entre dimensiones.

In [ ]:
import sys, os, pandas as pdif os.path.abspath("..") not in sys.path:    sys.path.insert(0, os.path.abspath(".."))from database.db import engineprint("Entorno listo.")

## 1. origen x timing

In [ ]:
df = pd.read_sql("SELECT origen_proceso, fecha_publicacion_estimada FROM ofertas", engine)df["fecha"] = pd.to_datetime(df["fecha_publicacion_estimada"])df["dia"] = df["fecha"].dt.day_name()print("Volumen por dia y origen:")print(pd.crosstab(df["origen_proceso"], df["dia"]))

## 2. origen x empresa

In [ ]:
query = """SELECT o.origen_proceso, e.nombre as empresaFROM ofertas o LEFT JOIN empresas e ON o.empresa_id = e.id"""df = pd.read_sql(query, engine)print("Empresas unicas por origen:")print(df.groupby("origen_proceso")["empresa"].nunique())

## 3. ingles x timing

In [ ]:
df = pd.read_sql("SELECT requiere_ingles, fecha_publicacion_estimada FROM ofertas", engine)df["fecha"] = pd.to_datetime(df["fecha_publicacion_estimada"])df["dia"] = df["fecha"].dt.day_name()print("Volumen por dia e ingles:")print(pd.crosstab(df["requiere_ingles"], df["dia"]))

## 4. experiencia x empresa

In [ ]:
query = """SELECT e.nombre as empresa, o.experiencia_aniosFROM ofertas o LEFT JOIN empresas e ON o.empresa_id = e.id"""df = pd.read_sql(query, engine).dropna(subset=["empresa"])print("Top empresas por experiencia promedio:")print(df.groupby("empresa")["experiencia_anios"].mean().round(2).sort_values(ascending=False).head(10))

## 5. experiencia x compatibilidad

In [ ]:
query = """SELECT o.experiencia_anios, c.scoreFROM ofertas o JOIN compatibilidades c ON o.id = c.oferta_id"""df = pd.read_sql(query, engine)print(f"Correlacion exp vs score: {df['experiencia_anios'].corr(df['score']):.4f}")bins = [0, 1.9, 4.9, 100]labels = ["Junior (0-2)", "Middle (2-5)", "Senior (5+)"]df["nivel"] = pd.cut(df["experiencia_anios"], bins=bins, labels=labels)print("Score promedio por nivel:")print(df.groupby("nivel")["score"].mean().round(4))

## 6. experiencia x timing

In [ ]:
df = pd.read_sql("SELECT experiencia_anios, fecha_publicacion_estimada FROM ofertas", engine)df["fecha"] = pd.to_datetime(df["fecha_publicacion_estimada"])df["dia"] = df["fecha"].dt.day_name()print("Experiencia promedio por dia:")print(df.groupby("dia")["experiencia_anios"].mean().round(2))

## 7. empresa x compatibilidad

In [ ]:
query = """SELECT e.nombre as empresa, c.scoreFROM ofertas oJOIN empresas e ON o.empresa_id = e.idJOIN compatibilidades c ON o.id = c.oferta_id"""df = pd.read_sql(query, engine)print("Top empresas por score:")print(df.groupby("empresa")["score"].mean().round(4).sort_values(ascending=False).head(10))

## 8. tech x compatibilidad

In [ ]:
query = """SELECT t.nombre as tech, c.scoreFROM ofertas_tecnologias otJOIN tecnologias t ON ot.tecnologia_id = t.idJOIN compatibilidades c ON ot.oferta_id = c.oferta_id"""df = pd.read_sql(query, engine)print("Score promedio por tecnologia (top 15):")print(df.groupby("tech")["score"].mean().round(4).sort_values(ascending=False).head(15))